In [2]:
import numpy as np
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# =========================
# LOAD LABELS
# =========================
df = pd.read_csv("features_summary/labels.csv")
label_names = sorted(df["label"].unique())

file_keys = df["file_key"].values
y = df["label_encoded"].values

# =========================
# CONFIG
# =========================
summary_dir = "features_summary"
mfcc_dir = "features_mfcc/flattened"
cqt_dir = "features_cqt/flattened"

# =========================
# BUILD DATASETS
# =========================
X_mfcc = []
X_cqt = []
X_all = []
y_filtered = []

for i, key in enumerate(file_keys):
    summary_path = os.path.join(summary_dir, f"{key}.npy")
    mfcc_path = os.path.join(mfcc_dir, f"{key}.npy")
    cqt_path = os.path.join(cqt_dir, f"{key}.npy")

    if not (os.path.exists(summary_path) and os.path.exists(mfcc_path) and os.path.exists(cqt_path)):
        continue

    summary_feat = np.load(summary_path)
    mfcc_feat = np.load(mfcc_path)
    cqt_feat = np.load(cqt_path)

    # -------------------------
    # Normalize EACH feature group
    # -------------------------
    summary_feat = (summary_feat - np.mean(summary_feat)) / (np.std(summary_feat) + 1e-6)
    mfcc_feat = (mfcc_feat - np.mean(mfcc_feat)) / (np.std(mfcc_feat) + 1e-6)
    cqt_feat = (cqt_feat - np.mean(cqt_feat)) / (np.std(cqt_feat) + 1e-6)

    # -------------------------
    # COMBINATIONS
    # -------------------------
    combined_mfcc = np.concatenate([summary_feat, mfcc_feat])
    combined_cqt = np.concatenate([summary_feat, cqt_feat])
    combined_all = np.concatenate([summary_feat, mfcc_feat, cqt_feat])

    X_mfcc.append(combined_mfcc)
    X_cqt.append(combined_cqt)
    X_all.append(combined_all)
    y_filtered.append(y[i])

X_mfcc = np.array(X_mfcc)
X_cqt = np.array(X_cqt)
X_all = np.array(X_all)
y_filtered = np.array(y_filtered)

print("MFCC shape:", X_mfcc.shape)
print("CQT  shape:", X_cqt.shape)
print("ALL  shape:", X_all.shape)

# =========================
# TRAIN / TEST SPLIT
# =========================
X_train_mfcc, X_test_mfcc, y_train, y_test = train_test_split(
    X_mfcc, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

X_train_cqt, X_test_cqt, _, _ = train_test_split(
    X_cqt, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

X_train_all, X_test_all, _, _ = train_test_split(
    X_all, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

# =========================
# SVM FUNCTION
# =========================
from sklearn.metrics import classification_report

def train_and_evaluate(X_train, X_test, y_train, y_test, name, label_names):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            C=10,
            gamma="scale",
            class_weight="balanced"
        ))
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # -------------------------
    # OVERALL METRICS
    # -------------------------
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')

    print(f"\n===== {name} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")

    # -------------------------
    # CLASSIFICATION REPORT
    # -------------------------
    print("\n--- Per-Class Metrics ---")
    print(classification_report(y_test, y_pred, target_names=label_names))

    # -------------------------
    # PER-CLASS ACCURACY (explicit)
    # -------------------------
    print("\n--- Accuracy per Instrument ---")
    for i, label in enumerate(label_names):
        idx = (y_test == i)
        correct = np.sum(y_pred[idx] == y_test[idx])
        total = np.sum(idx)

        acc_i = correct / total if total > 0 else 0
        print(f"{label}: {acc_i:.4f} ({correct}/{total})")

# =========================
# RUN EXPERIMENTS
# =========================
train_and_evaluate(X_train_mfcc, X_test_mfcc, y_train, y_test,
                   "SUMMARY + MFCC (PCA)", label_names)

train_and_evaluate(X_train_cqt, X_test_cqt, y_train, y_test,
                   "SUMMARY + CQT (PCA)", label_names)

train_and_evaluate(X_train_all, X_test_all, y_train, y_test,
                   "SUMMARY + MFCC + CQT (PCA)", label_names)

MFCC shape: (2365, 25)
CQT  shape: (2365, 16805)
ALL  shape: (2365, 16825)

===== SUMMARY + MFCC (PCA) =====
Accuracy : 0.8140
F1 Score : 0.8142
Precision: 0.8158
Recall   : 0.8140

--- Per-Class Metrics ---
              precision    recall  f1-score   support

         cel       0.81      0.74      0.77        78
         gac       0.79      0.85      0.82       127
         gel       0.90      0.86      0.88       152
         vio       0.74      0.76      0.75       116

    accuracy                           0.81       473
   macro avg       0.81      0.80      0.81       473
weighted avg       0.82      0.81      0.81       473


--- Accuracy per Instrument ---
cel: 0.7436 (58/78)
gac: 0.8504 (108/127)
gel: 0.8618 (131/152)
vio: 0.7586 (88/116)

===== SUMMARY + CQT (PCA) =====
Accuracy : 0.6300
F1 Score : 0.6166
Precision: 0.6288
Recall   : 0.6300

--- Per-Class Metrics ---
              precision    recall  f1-score   support

         cel       0.63      0.33      0.44        7

In [ ]:
import numpy as np
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

# =========================
# LOAD LABELS (UPDATED)
# =========================
labels_csv = "features_cqt/pca/labels.csv"
df = pd.read_csv(labels_csv)

# Use encoded labels directly
X_keys = df["file_key"].values
y = df["label_encoded"].values

# =========================
# CONFIG
# =========================
feature_root = "features_mfcc"
feature_types = ["flattened"]

kernels = ["rbf"]
C_values = [5]
gamma_values = ["scale"]

results = []

# =========================
# LOOP OVER FEATURE TYPES
# =========================
for ftype in feature_types:
    print(f"\n===== FEATURE TYPE: {ftype.upper()} =====")

    X = []
    y_filtered = []

    feature_dir = os.path.join(feature_root, ftype)

    for i, key in enumerate(X_keys):
        feature_path = os.path.join(feature_dir, f"{key}.npy")

        if not os.path.exists(feature_path):
            continue

        features = np.load(feature_path)

        X.append(features)
        y_filtered.append(y[i])

    X = np.array(X)
    y_filtered = np.array(y_filtered)

    print("X shape:", X.shape)
    print("y shape:", y_filtered.shape)

    # =========================
    # TRAIN / TEST SPLIT
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_filtered, test_size=0.20, random_state=42, stratify=y_filtered
    )

    best_f1 = 0
    best_config = None

    # =========================
    # GRID SEARCH (MANUAL LOOP)
    # =========================
    for kernel in kernels:
        for C in C_values:
            for gamma in gamma_values:

                # Skip gamma for linear kernel
                if kernel == "linear":
                    gamma = "auto"

                svm = Pipeline([
                    ("scaler", StandardScaler()),
                    ("clf", SVC(
                        kernel=kernel,
                        C=C,
                        gamma=gamma,
                        class_weight='balanced'
                    ))
                ])

                try:
                    svm.fit(X_train, y_train)

                    y_test_pred = svm.predict(X_test)

                    test_acc = accuracy_score(y_test, y_test_pred)
                    f1 = f1_score(y_test, y_test_pred, average='weighted')

                    print(f"[{ftype}] kernel={kernel}, C={C}, gamma={gamma} -> Acc={test_acc:.4f}, F1={f1:.4f}")

                    if f1 > best_f1:
                        best_f1 = f1
                        best_config = (kernel, C, gamma, test_acc)

                except Exception as e:
                    print("Skipped due to error:", e)

    print("\n>>> BEST RESULT FOR", ftype.upper())
    print(f"Kernel={best_config[0]}, C={best_config[1]}, gamma={best_config[2]}")
    print(f"Test Accuracy={best_config[3]:.4f}, F1={best_f1:.4f}")

    results.append((ftype, best_config, best_f1))


# =========================
# FINAL SUMMARY
# =========================
print("\n===== FINAL SUMMARY =====")
for r in results:
    ftype, config, f1 = r
    print(f"{ftype.upper():10s} | Kernel={config[0]} C={config[1]} gamma={config[2]} | F1={f1:.4f}")